# Блок 1. Практика: Настройка окружения и работа с данными

В этом notebook вы:
1. Настроите окружение для работы с данными FinanceFlow
2. Запустите базовую инфраструктуру через Docker Compose
3. Познакомитесь с данными через DuckDB и Parquet

> **Напоминание:** Убедитесь, что вы выполнили задания из README.md по настройке `.env` и загрузке данных через `bootstrap.py`.

## Часть 1: Проверка окружения

### 1.1 Проверка зависимостей

Сначала убедимся, что все необходимые библиотеки установлены.

In [ ]:
# Проверка установленных библиотек
try:
    import duckdb
    print("✓ DuckDB установлен")
except ImportError:
    print("✗ DuckDB не установлен. Запустите: uv sync")

try:
    import pyarrow
    import pyarrow.parquet as pq
    print("✓ PyArrow установлен")
except ImportError:
    print("✗ PyArrow не установлен. Запустите: uv sync")

try:
    import psycopg2
    print("✓ psycopg2 установлен")
except ImportError:
    print("✗ psycopg2 не установлен. Запустите: uv sync")

### 1.2 Проверка наличия данных

Проверим, что данные загружены через `bootstrap.py`.

In [ ]:
from pathlib import Path
import json

# Находим корень проекта
project_root = Path().resolve()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        raise RuntimeError("Не удалось найти корень проекта")

data_dir = project_root / "data"

# Проверяем наличие данных
print(f"Корень проекта: {project_root}")
print(f"Директория данных: {data_dir}")

# Проверяем версии
for version in ["lite", "full"]:
    version_dir = data_dir / version
    if version_dir.exists():
        print(f"\n✓ Версия {version} найдена")
        
        # Проверяем manifest.json
        manifest_path = version_dir / "manifest.json"
        if manifest_path.exists():
            with open(manifest_path) as f:
                manifest = json.load(f)
            print(f"  - Файлов в manifest: {len(manifest.get('files', []))}")
            print(f"  - Общий размер: {manifest.get('total_size', 0) / 1024 / 1024:.2f} MB")
        
        # Проверяем типы данных
        for data_type in ["auth_events", "nginx_logs", "dns_queries", "firewall_events"]:
            data_type_dir = version_dir / data_type
            if data_type_dir.exists():
                parquet_files = list(data_type_dir.glob("day=*/part-*.parquet"))
                print(f"  - {data_type}: {len(parquet_files)} файлов")
    else:
        print(f"\n✗ Версия {version} не найдена")
        if version == "lite":
            print("  Запустите: uv run python bootstrap.py --version lite")

## Часть 2: Docker Compose

### 2.1 Запуск инфраструктуры

Теперь запустим PostgreSQL через Docker Compose. Убедитесь, что вы создали `docker-compose.yml` в директории `lesson01/` согласно заданию 1.2 из README.md.

**Задание:** Выполните в терминале:

```bash
cd lesson01
docker compose up -d
```

После запуска проверьте статус:

```bash
docker compose ps
```

Вы должны увидеть контейнер `security-postgres` в статусе `running`.

### 2.2 Проверка подключения к PostgreSQL

**TODO:** Заполните параметры подключения из вашего `.env` файла и выполните ячейку ниже.

In [ ]:
# TODO: Заполните параметры подключения из .env
DB_HOST = "localhost"  # Измените при необходимости
DB_PORT = 5432
DB_NAME = "security_logs"  # Измените на значение из .env
DB_USER = "analyst"  # Измените на значение из .env
DB_PASSWORD = "your_password"  # Измените на значение из .env

try:
    import psycopg2
    
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    
    cursor = conn.cursor()
    cursor.execute("SELECT version();")
    result = cursor.fetchone()
    if result:
        print("✓ Подключение к PostgreSQL успешно")
        print(f"Версия: {result[0]}")
    
    cursor.close()
    conn.close()
except Exception as e:
    print(f"✗ Ошибка подключения: {e}")
    print("Проверьте, что:")
    print("1. Docker Compose запущен: docker compose ps")
    print("2. Параметры подключения корректны")
    print("3. База данных создана")

## Часть 3: Работа с данными через DuckDB

DuckDB — это аналитическая СУБД, которая отлично работает с Parquet-файлами. В отличие от PostgreSQL, DuckDB оптимизирован для аналитических запросов и может читать данные напрямую из файлов без предварительной загрузки.

### 3.1 Подключение к данным

Создадим подключение DuckDB и посмотрим на структуру данных.

In [ ]:
import duckdb

# Создаём подключение DuckDB
conn = duckdb.connect()

# TODO: Укажите версию данных (lite или full)
VERSION = "lite"  # Измените на "full" если загрузили полную версию

# Путь к данным
data_path = project_root / "data" / VERSION

print(f"Работаем с версией: {VERSION}")
print(f"Путь к данным: {data_path}")

# Проверяем наличие данных
if not data_path.exists():
    print(f"✗ Директория {data_path} не найдена!")
    print("Запустите: uv run python bootstrap.py --version lite")
else:
    print("✓ Данные найдены")

### 3.2 Изучение структуры данных: Auth Events

Начнём с событий аутентификации. Посмотрим на структуру таблицы и первые записи.

In [ ]:
# Читаем данные из Parquet
auth_path = data_path / "auth_events"

# DuckDB может читать все Parquet-файлы из директории с помощью wildcard
query = f"""
SELECT * 
FROM read_parquet('{auth_path}/day=*/part-*.parquet')
LIMIT 10
"""

result = conn.execute(query).fetchdf()
print("Первые 10 записей событий аутентификации:")
print(result)

### 3.3 Базовые запросы: Статистика по событиям

**TODO:** Напишите запрос для подсчёта общего количества событий аутентификации.

In [ ]:
# TODO: Напишите запрос для подсчёта общего количества событий
# Подсказка: используйте COUNT(*) и read_parquet с wildcard

query = """
-- Ваш запрос здесь
"""

try:
    result = conn.execute(query).fetchone()
    print(f"Всего событий аутентификации: {result[0]}")  # pyright: ignore[reportOptionalSubscript]
except Exception as e:
    print(f"Ошибка: {e}")
    print("Проверьте синтаксис запроса")

### 3.4 Группировка по типам событий

**TODO:** Напишите запрос для подсчёта событий по типам (login_success, login_failure, logout).

In [ ]:
# TODO: Напишите запрос для группировки по типам событий
# Подсказка: используйте GROUP BY и COUNT(*)

query = """
-- Ваш запрос здесь
"""

try:
    result = conn.execute(query).fetchdf()
    print("Статистика по типам событий:")
    print(result)
except Exception as e:
    print(f"Ошибка: {e}")
    print("Проверьте синтаксис запроса")

### 3.5 Работа с другими типами данных

Теперь попробуем работать с веб-логами (nginx_logs). Структура данных описана в `data/DATA_STRUCTURE.md`.

**TODO:** Напишите запрос для подсчёта запросов по HTTP-кодам статуса.

In [ ]:
# Путь к nginx логам
nginx_path = data_path / "nginx_logs"

# TODO: Напишите запрос для подсчёта запросов по статусам
# Подсказка: используйте GROUP BY status и COUNT(*)

query = f"""
-- Ваш запрос здесь
-- Используйте read_parquet('{nginx_path}/day=*/part-*.parquet')
"""

try:
    result = conn.execute(query).fetchdf()
    print("Статистика по HTTP-кодам:")
    print(result)
except Exception as e:
    print(f"Ошибка: {e}")
    print("Проверьте синтаксис запроса")

### 3.6 Изучение структуры: DNS запросы

Посмотрим на структуру DNS-запросов. DNS-логи содержат информацию о доменах, к которым обращались пользователи.

**TODO:** Напишите запрос для поиска топ-10 самых часто запрашиваемых доменов.

In [ ]:
# Путь к DNS логам
dns_path = data_path / "dns_queries"

# TODO: Напишите запрос для поиска топ-10 доменов
# Подсказка: используйте GROUP BY domain, COUNT(*) и ORDER BY ... DESC LIMIT 10

query = f"""
-- Ваш запрос здесь
-- Используйте read_parquet('{dns_path}/day=*/part-*.parquet')
"""

try:
    result = conn.execute(query).fetchdf()
    print("Топ-10 самых часто запрашиваемых доменов:")
    print(result)
except Exception as e:
    print(f"Ошибка: {e}")
    print("Проверьте синтаксис запроса")

## Итоги

В этом блоке вы:

1. ✅ Настроили окружение и загрузили данные через `bootstrap.py`
2. ✅ Запустили базовую инфраструктуру через Docker Compose
3. ✅ Познакомились с DuckDB и работой с Parquet-файлами
4. ✅ Выполнили базовые запросы для изучения структуры данных

### Что дальше?

В следующих блоках вы:

**Блок 2: Сбор логов в реальном времени**
- Настроите Vector для сбора логов из различных источников
- Научитесь парсить разные форматы логов (JSON, Combined Log Format, BIND, CEF)
- Настроите маршрутизацию данных в PostgreSQL

**Блок 3: Организация хранения данных**
- Создадите оптимизированные индексы для быстрых запросов
- Научитесь экспортировать данные в Parquet для аналитики
- Поймёте разницу между OLTP (PostgreSQL) и OLAP (DuckDB)

**Блок 4: SQL-аналитика для ИБ**
- Научитесь искать индикаторы атаки с помощью SQL
- Построите timeline инцидента
- Найдёте следы brute-force, lateral movement и других типов атак

**Блок 5: Углубленный анализ с Python**
- Используете Polars для обработки больших объёмов данных
- Примените ML для поиска аномалий
- Обнаружите DGA-домены и другие сложные паттерны

**Блок 6: Визуализация в Grafana**
- Создадите дашборды для мониторинга безопасности
- Настроите алерты на подозрительную активность
- Визуализируете timeline атаки

> **Интрига:** Помните алерт, о котором говорила Марина в начале дня? В данных, которые вы только что изучили, уже есть следы подозрительной активности. В блоках 4-5 вы научитесь находить эти индикаторы и строить полную картину инцидента. Но сначала нужно настроить сбор новых логов — этим мы займёмся в следующем блоке.

### Очистка

После завершения работы можно остановить Docker Compose:

```bash
cd lesson01
docker compose down
```

Для полной очистки (включая тома):

```bash
docker compose down -v
```